# DINOv3-Guided YOLO26 SSL for Football Images

This tutorial uses the reusable ssl-detection-lab DINOv3 module and LightlyTrain Distillation v3 to transfer a frozen DINOv3 ViT-B/16 teacher into a YOLO26 student without using football labels.

**Learning goals**

- Validate the local DINOv3 ViT-B/16 checkpoint
- Understand why DINOv3 weights cannot be copied directly into YOLO26
- Run global and spatial Distillation v3 through a reusable library call
- Train with mixed precision on a Kaggle T4 GPU
- Export an Ultralytics-compatible YOLO26 checkpoint
- Visualize the student representation with t-SNE


## What is trained

DINOv3 and YOLO26 use different architectures, so their tensor weights are not directly interchangeable. LightlyTrain keeps the DINOv3 teacher frozen and trains YOLO26 using Distillation v3 global and spatial objectives.

The reusable module passes the local checkpoint through `teacher_weights`, validates `dinov3/vitb16`, and returns `exported_models/exported_last.pt` for downstream fine-tuning.

This is DINOv3-guided YOLO feature distillation. It is not a reproduction of full DINOv3 pretraining, which uses DINO self-distillation, iBOT, KoLeo regularization, Gram anchoring, and large-scale distributed training.


Official resources: https://github.com/facebookresearch/dinov3 and https://github.com/facebookresearch/dinov3/blob/main/LICENSE.md

DINOv3 uses a separate DINOv3 License. Obtain an authorized official weight URL or download the checkpoint and attach it to Kaggle. A local checkpoint is recommended because signed URLs can expire and should not be saved in public notebook outputs.


## Roadmap

1. Configure Kaggle and the teacher checkpoint
2. Inspect the football dataset
3. Validate DINOv3 features
4. Configure YOLO26 distillation
5. Preview the label-free input
6. Train with mixed precision
7. Review loss and checkpoints
8. Extract student features
9. Plot t-SNE and nearest neighbors
10. Complete an exercise


## 1. Configure Kaggle

Select a GPU accelerator, enable Internet access for package installation, and attach the football dataset and DINOv3 ViT-B/16 weights as Kaggle inputs.


In [ ]:
%pip install -q -U "lightly-train[ultralytics]>=0.16.2"
%pip install -q --no-cache-dir --force-reinstall --no-deps "git+https://github.com/rifat963/ssl-detection-lab.git@main"


In [ ]:
DINOV3_MODEL = "dinov3/vitb16"
DINOV3_WEIGHTS = "/kaggle/input/datasets/mrifatrashid/dinov3-weigths/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"
YOLO_STUDENT = "ultralytics/yolo26n.yaml"


The configuration requires the attached local DINOv3 ViT-B/16 checkpoint. It does not silently fall back to downloaded teacher weights.


In [ ]:
from pathlib import Path
from collections import Counter
from importlib.metadata import version as installed_version
import json
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
import yaml
from PIL import Image
from packaging.version import Version
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from tqdm.auto import tqdm
from ultralytics import YOLO
import lightly_train

assert Version(installed_version("ssl-detection-lab")) >= Version("0.8.0")

import ssldet
from ssldet import (
    DINOv3YOLOConfig,
    pretrain_dinov3_yolo26,
    validate_dinov3_yolo26_support,
)
from ssldet.backbones import YOLOBackboneEncoder
from ssldet.data import IMAGENET_MEAN, IMAGENET_STD, UnlabeledImageDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")
sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings(
    "ignore",
    message=r".*isinstance\(treespec, LeafSpec\).*deprecated.*",
    category=FutureWarning,
)

assert torch.cuda.is_available(), "Select a GPU accelerator before continuing."
if not DINOV3_WEIGHTS.startswith(("http://", "https://")):
    assert Path(DINOV3_WEIGHTS).is_file(), "Update DINOV3_WEIGHTS to the attached checkpoint"

DEVICE = torch.device("cuda:0")
pd.Series({
    "ssl-detection-lab": ssldet.__version__,
    "PyTorch": torch.__version__,
    "GPU": torch.cuda.get_device_name(0),
    "LightlyTrain": getattr(lightly_train, "__version__", "unknown"),
    "teacher model": DINOV3_MODEL,
    "teacher weights": DINOV3_WEIGHTS,
})


## 2. Inspect the football dataset


In [ ]:
DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/iasadpanwhar/football-player-detection-yolov8/football_players_detection/football_players_detection"),
    Path("/kaggle/input/football-player-detection-yolov8/football_players_detection/football_players_detection"),
]
DATASET_ROOT = next((path for path in DATASET_CANDIDATES if path.is_dir()), None)
if DATASET_ROOT is None:
    raise FileNotFoundError("Attach the football-player-detection-yolov8 dataset")

SPLITS = {
    split: {
        "images": DATASET_ROOT / split / "images",
        "labels": DATASET_ROOT / split / "labels",
    }
    for split in ("train", "valid", "test")
}
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_files(directory):
    return sorted(
        path for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )

summary = []
for split, paths in SPLITS.items():
    summary.append({
        "split": split,
        "images": len(image_files(paths["images"])),
        "labels": len(list(paths["labels"].glob("*.txt"))),
    })
pd.DataFrame(summary).set_index("split")


In [ ]:
train_images = image_files(SPLITS["train"]["images"])
sample_paths = random.Random(SEED).sample(train_images, min(8, len(train_images)))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for axis in axes.flat:
    axis.axis("off")
for axis, path in zip(axes.flat, sample_paths):
    with Image.open(path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(path.name, fontsize=9)
plt.tight_layout()
plt.show()


## 3. Validate DINOv3 and YOLO26 support

The module checks the current LightlyTrain catalog before starting a long run.


In [ ]:
support = validate_dinov3_yolo26_support(lightly_train)
pd.Series({
    "teacher available": support[DINOV3_MODEL],
    "student available": support[YOLO_STUDENT],
    "distillation available": support["distillation"],
    "teacher weights MB": round(Path(DINOV3_WEIGHTS).stat().st_size / 1024 ** 2, 2),
})


In [ ]:
torch.cuda.empty_cache()


## 4. Configure DINOv3-to-YOLO26 Distillation v3

The fast preset uses two epochs for a pipeline check. Use the full preset for a meaningful representation-learning experiment.


In [ ]:
FAST_RUN = True
OUTPUT_DIR = Path("/kaggle/working/dinov3_yolo26_football")
config = DINOv3YOLOConfig(
    data=str(SPLITS["train"]["images"]),
    output_dir=str(OUTPUT_DIR),
    teacher_weights=DINOV3_WEIGHTS,
    teacher_model=DINOV3_MODEL,
    student_model=YOLO_STUDENT,
    epochs=2 if FAST_RUN else 100,
    batch_size=8 if FAST_RUN else 32,
    image_size=224,
    num_workers=2,
    devices=1,
    accelerator="gpu",
    precision="16-mixed",
    seed=SEED,
    extra_arguments={"trainer_args": {"enable_model_summary": False}},
).validate()

pd.Series({
    "preset": "fast" if FAST_RUN else "full",
    "student": config.student_model,
    "teacher": config.teacher_model,
    "epochs": config.epochs,
    "unlabelled images": len(train_images),
    "batch size": config.batch_size,
    "image size": config.image_size,
    "precision": config.precision,
})


## 5. Preview the label-free training input


In [ ]:
preview_transform = v2.Compose([
    v2.Resize((config.image_size, config.image_size), antialias=True),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
preview_dataset = UnlabeledImageDataset(train_images, preview_transform)
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for index, axis in enumerate(axes):
    tensor = preview_dataset[index]
    axis.imshow((tensor * std + mean).clamp(0, 1).permute(1, 2, 0))
    axis.axis("off")
plt.tight_layout()
plt.show()


## 6. Run DINOv3 distillation


In [ ]:
training_result = pretrain_dinov3_yolo26(config, module=lightly_train)
training_result


LightlyTrain loads the local frozen DINOv3 teacher and exports the trained YOLO26 student to `exported_models/exported_last.pt`. The teacher checkpoint is not copied into the exported detector. A progress line such as `185/1288` means training is running; a real failure ends with an exception traceback.


## 7. Review loss and checkpoints


In [ ]:
YOLO_CHECKPOINT = training_result.yolo_checkpoint
LIGHTLY_CHECKPOINT = training_result.training_checkpoint
metrics_records = []
if training_result.metrics_jsonl.is_file():
    for line in training_result.metrics_jsonl.read_text().splitlines():
        if line.strip():
            metrics_records.append(json.loads(line))
metrics_frame = pd.json_normalize(metrics_records)
if not metrics_frame.empty:
    display(metrics_frame.tail())

pd.Series({
    "Lightly checkpoint": str(LIGHTLY_CHECKPOINT),
    "YOLO student checkpoint": str(YOLO_CHECKPOINT),
    "teacher": training_result.teacher_model,
    "teacher weights": str(training_result.teacher_weights),
    "unlabelled images": len(train_images),
})


## 8. Extract YOLO student validation features


In [ ]:
yaml_candidates = sorted(DATASET_ROOT.parent.rglob("data.yaml"))
dataset_yaml = yaml_candidates[0] if yaml_candidates else None
metadata = yaml.safe_load(dataset_yaml.read_text()) if dataset_yaml else {}
raw_names = metadata.get("names", {})
if isinstance(raw_names, list):
    CLASS_NAMES = {index: name for index, name in enumerate(raw_names)}
elif isinstance(raw_names, dict):
    CLASS_NAMES = {int(index): name for index, name in raw_names.items()}
else:
    CLASS_NAMES = {}

def object_classes(label_path):
    if not label_path.exists():
        return []
    return [
        int(float(line.split()[0]))
        for line in label_path.read_text().splitlines()
        if line.strip()
    ]

validation_images = image_files(SPLITS["valid"]["images"])
class_frequency = Counter(
    class_id
    for path in validation_images
    for class_id in object_classes(SPLITS["valid"]["labels"] / f"{path.stem}.txt")
)
if not CLASS_NAMES:
    CLASS_NAMES = {class_id: f"class {class_id}" for class_id in class_frequency}

def image_label(path):
    values = set(object_classes(SPLITS["valid"]["labels"] / f"{path.stem}.txt"))
    return min(values, key=lambda value: class_frequency[value]) if values else -1


In [ ]:
selected_paths = validation_images
if len(selected_paths) > 500:
    selected_paths = sorted(random.Random(SEED).sample(selected_paths, 500))
feature_transform = v2.Compose([
    v2.Resize(config.image_size + 32, antialias=True),
    v2.CenterCrop(config.image_size),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class FeatureDataset(Dataset):
    def __len__(self):
        return len(selected_paths)

    def __getitem__(self, index):
        path = selected_paths[index]
        with Image.open(path) as image:
            tensor = feature_transform(image.convert("RGB"))
        return tensor, image_label(path)

loader = DataLoader(FeatureDataset(), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
encoder = YOLOBackboneEncoder(YOLO(str(YOLO_CHECKPOINT)).model).to(DEVICE).eval()
feature_batches = []
label_batches = []
with torch.inference_mode(), torch.amp.autocast("cuda"):
    for images, labels in tqdm(loader, desc="Extracting DINOv3-guided YOLO features"):
        features = encoder(images.to(DEVICE, non_blocking=True))
        feature_batches.append(F.normalize(features.float(), dim=1).cpu())
        label_batches.append(labels.numpy())
feature_matrix = torch.cat(feature_batches)
label_ids = np.concatenate(label_batches)

pd.Series({
    "images": len(feature_matrix),
    "dimensions": feature_matrix.shape[1],
    "mean feature standard deviation": feature_matrix.std(dim=0).mean().item(),
})


## 9. Plot t-SNE


In [ ]:
perplexity = min(30.0, max(2.0, (len(feature_matrix) - 1) / 3))
coordinates = TSNE(
    n_components=2,
    perplexity=perplexity,
    learning_rate="auto",
    init="pca",
    max_iter=1000,
    random_state=SEED,
).fit_transform(feature_matrix.numpy())
plot_frame = pd.DataFrame({
    "t-SNE 1": coordinates[:, 0],
    "t-SNE 2": coordinates[:, 1],
    "class": [CLASS_NAMES.get(int(value), "unlabelled") for value in label_ids],
})
fig, axis = plt.subplots(figsize=(12, 8))
sns.scatterplot(
    data=plot_frame,
    x="t-SNE 1",
    y="t-SNE 2",
    hue="class",
    palette="tab10",
    s=65,
    alpha=0.82,
    ax=axis,
)
axis.set_title("t-SNE of DINOv3-guided YOLO26 features")
axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Nearest neighbors


In [ ]:
QUERY_INDEX = 0
similarities = feature_matrix @ feature_matrix[QUERY_INDEX]
neighbors = torch.topk(similarities, k=min(6, len(feature_matrix))).indices.tolist()
neighbors = [value for value in neighbors if value != QUERY_INDEX][:5]
display_indices = [QUERY_INDEX] + neighbors
fig, axes = plt.subplots(1, len(display_indices), figsize=(4 * len(display_indices), 4))
for position, (axis, index) in enumerate(zip(axes, display_indices)):
    with Image.open(selected_paths[index]) as image:
        axis.imshow(image.convert("RGB"))
    title = "Query" if position == 0 else f"Similarity {similarities[index]:.3f}"
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()


## Exercise

Create a second run from `ultralytics/yolo26n.pt` instead of the YAML architecture. Keep every other setting fixed and compare t-SNE neighborhoods and downstream detection mAP.


In [ ]:
exercise_config = DINOv3YOLOConfig(
    **{
        **config.to_dict(),
        "student_model": "ultralytics/yolo26n.pt",
        "output_dir": "/kaggle/working/dinov3_yolo26_coco_initialized",
    }
).validate()
pd.Series({
    "student": exercise_config.student_model,
    "teacher": exercise_config.teacher_model,
    "output": exercise_config.output_dir,
})


## Practical checks

- The fast preset uses batch size 8; reduce it to 4 if CUDA memory is exhausted.
- A two-epoch fast run validates the pipeline but is not a meaningful SSL experiment.
- A one-time scheduler-order warning can occur when AMP skips an optimizer update. Continue if loss remains finite and progress advances; switch `precision` to `32-true` only if the warning repeats with NaN or infinite loss.
- The teacher checkpoint and DINOV3_MODEL must match.
- Keep the official teacher frozen.
- The module deliberately raises an error when the local teacher checkpoint is missing or incompatible.
- Distillation loss is not detection accuracy; fine-tune and evaluate the exported YOLO student downstream.
